# Retrain StressTwin-F and export loadable weights (Kaggle, CPU, Internet ON)

**Why:** your reproducibility package has no saved weights and its training script saves none. This notebook retrains the
published configuration (top-2 RoBERTa layers + head, fidelity loss, 5 seeds; plus 5 plain seeds for the ablation) with the
same code and seeds, saving the weights, and checks them against the released results.

**Before running:** Add Input: your private datasets `stresstwin-p` (this package) and `stresstwin-f-repro` (upload
`stresstwin_f_reproducibility_package_2.zip` as a private dataset). Settings: Accelerator None (the training code is CPU),
Internet On. Use *Save & Run All (Commit)* so it runs unattended.

**Time:** not measured. The package README gives about 6 hours for its whole pipeline on 2 CPU cores; this notebook runs only a
subset, so expect a few hours. Finished seeds are skipped on re-run (a seed counts as done only when its weights file exists).

If any cell fails, read the traceback and check the data paths before re-running.

In [ ]:
import os, sys, glob, shutil, zipfile, subprocess
assert sys.version_info >= (3, 11), sys.version
WORK = '/kaggle/working'
zips = glob.glob('/kaggle/input/**/*reproducibility_package*.zip', recursive=True)
# Kaggle may auto-extract an uploaded zip: accept either the zip or an already-extracted folder that has requirements.txt + src/
src = None
for r, d, f in os.walk('/kaggle/input'):
    if 'requirements.txt' in f and os.path.isdir(os.path.join(r, 'src')) and os.path.isdir(os.path.join(r, 'scripts')):
        src = r; break
assert zips or src, "add the dataset that holds stresstwin_f_reproducibility_package_2 (zip or extracted)"
if not os.path.exists(f'{WORK}/stresstwin'):
    if zips:
        zipfile.ZipFile(zips[0]).extractall(WORK)
    else:
        shutil.copytree(src, f'{WORK}/stresstwin')
REPO = f'{WORK}/stresstwin'; assert os.path.exists(f'{REPO}/requirements.txt'), os.listdir(WORK)
KIT = None
for r, d, f in os.walk('/kaggle/input'):
    if os.path.basename(r) == 'retrain_kit' and '03_train_eval.py' in f:
        KIT = r; break
assert KIT, "retrain_kit not found: add the stresstwin-p dataset (updated package)"
PKG = os.path.dirname(KIT)
if not os.path.exists(f'{WORK}/pkg'): shutil.copytree(PKG, f'{WORK}/pkg')
def sh(cmd, check=True):
    print('$', cmd); r = subprocess.run(cmd, shell=True, cwd=REPO); 
    assert r.returncode == 0 or not check, f'command failed: {cmd}'
print('repo', REPO, '| kit', KIT)

In [ ]:
# isolated environment with the package's pinned versions (torch CPU build to keep it small)
sh(f'{sys.executable} -m venv {WORK}/venv')
PY = f'{WORK}/venv/bin/python'
sh(f'{PY} -m pip -q install --upgrade pip')
sh(f'{PY} -m pip -q install torch==2.5.1 --index-url https://download.pytorch.org/whl/cpu')
reqs = [l for l in open(f'{REPO}/requirements.txt').read().splitlines() if l.strip() and not l.lstrip().startswith('#') and not l.startswith('torch')]
open(f'{WORK}/reqs_notorch.txt', 'w').write('\n'.join(reqs))
sh(f'{PY} -m pip -q install -r {WORK}/reqs_notorch.txt')
sh(f'{PY} -m pip -q install -e {WORK}/pkg')
sh(f'{PY} -c "import torch, transformers, numpy, pandas; print(torch.__version__, transformers.__version__, numpy.__version__, pandas.__version__)"')

In [ ]:
# data (checksums verified), then the RoBERTa-base backbone from the spaCy wheel (as in the published study)
sh('bash scripts/download_data.sh')
sh(f'{PY} scripts/00_prepare_backbone.py')

In [ ]:
# one-off frozen-prefix encoding of all texts and perturbed views
sh(f'NTHREADS=4 {PY} scripts/01_build_cache.py')

In [ ]:
# keep the RELEASED results for comparison, then retrain with weights saved. Settings = scripts/run_final.sh.
shutil.copytree(f'{REPO}/results', f'{WORK}/results_released', dirs_exist_ok=True)
shutil.copy(f'{KIT}/03_train_eval.py', f'{REPO}/scripts/03_train_eval.py')
C = '--epochs 3 --lr 1e-4 --threads 4 --ckpt_dir results/ckpt'
for s in range(5):
    sh(f'{PY} scripts/03_train_eval.py --name stresstwin_f --seeds {s} {C} --fidelity --lam_cc 0.5 --lam_st 0.5')
for s in range(5):
    sh(f'{PY} scripts/03_train_eval.py --name plain_ft --seeds {s} {C}')

In [ ]:
# retrained vs released test F1, per seed (expect small differences from hardware; large ones mean something is different)
import json
for name in ('plain_ft', 'stresstwin_f'):
    for s in range(5):
        new = json.load(open(f'{REPO}/results/runs/{name}_s{s}.json'))['test']['f1']
        old = json.load(open(f'{WORK}/results_released/runs/{name}_s{s}.json'))['test']['f1']
        print(f'{name:13s} seed {s}: released {old:.4f}  retrained {new:.4f}  diff {new-old:+.4f}')

In [ ]:
# the scorer used by stresstwin_p must reproduce the logits this run stored
for name in ('stresstwin_f', 'plain_ft'):
    sh(f'{PY} {KIT}/verify_export.py --backbone artifacts/roberta_backbone --ckpt_dir results/ckpt --name {name}')

In [ ]:
# export: this folder becomes the private dataset "stresstwin-f-ckpts" (about 1-2 GB)
EXP = f'{WORK}/stf_export'; os.makedirs(EXP, exist_ok=True)
shutil.copytree(f'{REPO}/artifacts/roberta_backbone', f'{EXP}/backbone', dirs_exist_ok=True)
shutil.copytree(f'{REPO}/results/ckpt', f'{EXP}/ckpt', dirs_exist_ok=True)
shutil.copytree(f'{REPO}/results/runs', f'{EXP}/runs', dirs_exist_ok=True)
shutil.copy(f'{REPO}/results/backbone_provenance.json', EXP)
sh(f'du -sh {EXP}/*')
print('Next: Save Version, then Output > New Dataset from stf_export, private, named stresstwin-f-ckpts.')